# §68 — Edge fill on the CLIP-only DoTA host: **L5dfiDoTAef** vs the trunk L0dfiDoTArr (s2024)

Rule: `core/docs/final/KIM_RUN.md` **§68**, written before any number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **Needs §64's DoTA host done** (`kim_clip_host_dota.ipynb`) and §67 (`kim_window_geometry.ipynb`). PRELIMINARY (n = 1). **K5f stays the reference. DoTA-CAP-dev is in-domain for every row.**

| row | = | here |
|---|---|---|
| **L5dfiDoTAef** | L5dfiDoTA + `kim.edge_fill=true` (no constant at the sequence edges, replicate-padded gates) | train + read |
| L0dfiDoTArr | trunk control (§64.2, Tesla T4) | reused; if the start differs → **L0dfiDoTArr2** re-trained here and used |
| L5dfiDoTA | the failed row (§64.2) | reused, printed (F) |

**E** = L5dfiDoTAef − trunk decides (DoTA-CAP-dev + DADA); **F** = L5dfiDoTAef − L5dfiDoTA printed; slot check from §67's window read of the new row.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-4 | setup, deps, helpers, preflight | no | 3-5 min |
| 5 | tests — **raises if red** | no | 2 min |
| 6 | stage corpus, plain CLIP, KNN | no | 10-15 min |
| 7 | train L5dfiDoTAef (1,764 steps) + start gate (→ L0dfiDoTArr2 if needed) | **yes** | ≈ L5dfiDoTA's time (×2 if the trunk is re-trained) |
| 8 | reads: DoTA-CAP-dev + DADA | yes | 15-25 min per row |
| 9 | window-geometry read of the new row (§67 tool) | yes | 10-20 min |
| 10 | report -> **paste `REPORTS/kim_edge_fill_report.md` to Claude** | no | 3-5 min |

A re-run skips what is on Drive. **Watch the log: past 2 s/step, stop and paste it.** Thesis / Thesis-V2 are only read.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Paths, rows, inputs

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, Nexar caches, A2 bakes, KNN, census
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, earlier rows and reads; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 68. Edge fill' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §68 -- re-upload kat-vad/'
assert (REPO / 'core/tools/kim_edge_fill_report.py').is_file(), 'no kim_edge_fill_report -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'KIM_EDGE_FILL_ROW'), 'constants has no §68 rows -- re-upload kat-vad/'

K5       = constants.KIM_MAIN_ARM                  # "K5": names run / eval sub-folders only
SEED     = constants.KIM_SEED                      # 2024
OFF      = constants.V2_OFF
ENC      = constants.V5_ENCODER
IGN      = constants.DVS_ANCHOR_MODE_IGNORE
DATASET  = constants.NEXAR_DATASET
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
STRIDE_B = constants.V2_E1_STRIDE_BC               # DoTA s1[::3] = protocol B rows
KIM_FLAGS = constants.KIM_ABLATION_FLAGS           # the four K5dfi KIM flags
CAP_FLAGS = constants.KIM_CLIP_HOST_CAP_FLAGS      # Kcap: enabled, prototypes, capacity_control
RR       = constants.KIM_ABLATION_RERUN_SUFFIX
ROOT     = constants.OUTPUT_ROOT / constants.KIM_CLIP_HOST_OUTPUT_DIRNAME   # L5 / L5cap (+ L0 re-run)
REPORTS  = ROOT / 'REPORTS'
V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
WINDOW   = Path(DRIVE_V2) / 'data' / f'{DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'   # defs of every read
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
T2_CLIP  = Path(DRIVE) / 'cache' / 'clip' / T2NAME                 # plain CLIP: every L row's DADA input
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'          # plain CLIP: every L row's DoTA input

HOST     = constants.KIM_CLIP_HOST_DOTA
NAMES    = constants.KIM_CLIP_HOST_ROWS[HOST]      # L5dfiDoTA, L5dficapDoTA, L0dfiDoTA, K5dfiDoTA, K0dfiDoTA
DOTA_ROOT = constants.OUTPUT_ROOT / constants.KIM_DOTA_TRAIN_OUTPUT_DIRNAME     # §55 / §58 rows
BUILD    = DOTA_ROOT / 'REPORTS' / constants.DOTA_TRAIN_BUILD_REPORT
DATA     = Path(DRIVE_F) / 'data' / constants.DOTA_TRAIN_CORPUS_DIRNAME
CLIP_S3  = constants.CLIP_CACHE_DIR / constants.DOTA_TRAIN_CLIP_DATASET        # plain CLIP s3 of DoTA train (§55)
KNN      = constants.CACHE_ROOT / 'knn' / DATA.name / constants.KNN_CACHE_FILENAME
EF       = constants.KIM_EDGE_FILL_KEY
NEW      = constants.KIM_EDGE_FILL_ROW                                         # L5dfiDoTAef
FAILED   = ROOT / constants.KIM_EDGE_FILL_FAILED_ROW                           # L5dfiDoTA (§64.2)
TRUNK    = ROOT / constants.KIM_EDGE_FILL_TRUNK_ROW                            # L0dfiDoTArr (§64.2)
TRUNK2   = ROOT / constants.KIM_EDGE_FILL_TRUNK_RERUN                          # L0dfiDoTArr2 if needed
CAP_ROW  = ROOT / NAMES['L5cap']                                               # L5dficapDoTA, for §67's read
LOCAL    = Path('/content/kim_edge_fill_dota')
LOCAL.mkdir(parents=True, exist_ok=True)

INPUTS = [('§55 build report', BUILD), ('§55 corpus windows', DATA / constants.WINDOWS_FILENAME), ('§55 KNN', KNN),
          ('§55 plain CLIP s3', CLIP_S3), ('Nexar_window defs', WINDOW / constants.DEFS_FILENAME), ('T2 meta', T2 / constants.META_FILENAME),
          ('T2 census', T2_CENSUS), ('T2 plain CLIP', T2_CLIP), ('DADA annotation', ANNOT), ('DoTA s1', DOTA_S1),
          ('DoTA metadata', DOTA / 'metadata_val.json')]
for root in (FAILED, TRUNK, CAP_ROW):
    INPUTS += [(f'{root.name} verified', root / K5 / f's{SEED}' / 'verified.json'),
               (f'{root.name} DoTA read', root / 'eval' / 'dota' / K5 / 'clip_scores.npz'),
               (f'{root.name} DADA read', root / 'eval' / 'dada' / K5 / 'clip_scores.npz')]
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing (run kim_clip_host_dota.ipynb first?) -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | new row {NEW} -> {ROOT}')

## 3. Dependencies

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- training and reads need one'
df -h /content | tail -1

## 4. Helpers + preflight (CPU)

In [ ]:
import torch  # noqa: E402
import yaml  # noqa: E402

from core.config import Config                         # noqa: E402
from core.data.v2_splits import load_split             # noqa: E402
from core.tools import k5_edge_report as edge          # noqa: E402
from core.tools import kim_ablation_report as ka       # noqa: E402
from core.tools.dada_whole_eval import test_sources    # noqa: E402
from core.tools.v5_stage1 import _flat                 # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    names = [f.name for f in src.iterdir() if f.is_file()] if ids is None else \
        [f'{v}.npy' for v in ids] + [m.name for p in ('*.json', '*.npz') for m in src.glob(p)]
    todo = [n for n in names if not (dst / n).exists()]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda n: shutil.copy2(src / n, dst / n), todo))
    if todo:
        print(f'  staged {src.name}: {len(todo)} files in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(run_dir, steps):
    rec, ckpt = run_dir / 'verified.json', run_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def cfg_of(root):
    return _flat(yaml.safe_load((root / K5 / f's{SEED}' / 'config.yaml').read_text()))


def run_dir(root):
    return root / K5 / f's{SEED}'


def last_epoch(rows, key):
    last = [r for r in rows if r['epoch'] == rows[-1]['epoch']]
    return sum(r.get(key, float('nan')) for r in last) / len(last)


def check_run(root, kind):
    """J10, §31 floor, EF4 config diff. ``kind`` in {'ef', 'L0rr2'}."""
    rd = run_dir(root)
    rows = metrics_rows(rd / 'metrics.jsonl')
    assert ckpt_step(rd / 'checkpoint_last.pt') == rows[-1]['global_step'] == STEPS, f'{root.name}: J10'
    got = cfg_of(root)
    base = failed_cfg if kind == 'ef' else trunk_cfg
    diff = ka.config_diff(base, got)
    want = {EF: (False, True)} if kind == 'ef' else {}
    assert diff == want, f'{root.name}: diff {diff}, §68 EF4 wants {want}'
    print(f'{root.name}: config diff vs {"L5dfiDoTA" if kind == "ef" else TRUNK.name}: {diff}')
    mul = last_epoch(rows, 'mul_mil')
    assert mul > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{root.name}: last-epoch mul_mil {mul:.4f} -- paste to Claude'
    keys = ('mil', 'mul_mil', 'kim_mil', 'kim_nkc')
    print('  last epoch: ' + ' | '.join(f'{k} {last_epoch(rows, k):.4f}' for k in keys))


def train_row(name, sets, kind):
    """Train one row into ROOT/<name> (resumable, atomic publish, J10, env recorded)."""
    root = ROOT / name
    rd, ckpt = run_dir(root), run_dir(root) / 'checkpoint_last.pt'
    if is_verified(rd, STEPS):
        print(f'{name} s{SEED}: verified on Drive -- not retrained')
        check_run(root, kind)
        return root
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / name
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    env = {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': str(torch.version.cuda)}
    print(f'{name}: trains on {env}')
    (local / constants.KIM_RUN_ENV_FILENAME).write_text(json.dumps(env))
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in sets for x in ('--set', s)], '--data-dir', L_DATA,
         '--clip-dir', L_TRAIN_X, '--output-dir', local, '--knn-cache', L_KNN, *resume], log=LOCAL / f'train_{name}.log')
    last = metrics_rows(local / 'metrics.jsonl')[-1]
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{name}: J10 locally {last["global_step"]}'
    for f in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml', constants.KIM_RUN_ENV_FILENAME):
        publish(local / f, rd / f)
    flush_drive()
    assert ckpt_step(ckpt) == STEPS, f'{name}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': ckpt.stat().st_size}))
    publish(local / 'verified.json', rd / 'verified.json')
    print(f'{name}: trained in {(time.time() - t0) / 60:.0f} min | step-1 mil {metrics_rows(local / "metrics.jsonl")[0]["mil"]:.6f}')
    check_run(root, kind)
    return root


def start(ctrl_root, new_root):
    st = edge.start_check(run_dir(ctrl_root), run_dir(new_root))
    print(f"CH3 step 1: {new_root.name} {st['new']:.6f} vs {ctrl_root.name} {st['control']:.6f} -> "
          f"{'matched' if st['same_draws'] else 'DIFFERENT START'}")
    return st['same_draws']


def read_dota(root):
    """DoTA-CAP-dev, protocol B on plain DoTA_s1_ncc[::3] (every L row)."""
    out = root / 'eval' / 'dota' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / root.name / 'dota'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', run_dir(root) / 'checkpoint_last.pt',
             '--input-dir', L_DOTA_S1, '--s1-dir', L_DOTA_S1, '--metadata', DOTA / 'metadata_val.json',
             '--split-file', DOTA / 'val_split.txt', '--data-dir', L_LAB, '--split', CAP_DEV, '--stride', STRIDE_B,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{root.name}_dota.log')
        publish(local, out)
    r = json.loads((out / 'protocol_b_readout.json').read_text())
    assert r['clips'] == len(CAP_IDS) and r['global_steps'] == {f's{SEED}': STEPS} and r['arm']['motion'] == OFF, r
    print(f"{root.name} DoTA-CAP-dev: macro {r['macro']['mean']:.4f}")


def read_dada(root):
    """DADA whole on plain DADA2000_orig CLIP (every L row)."""
    out = root / 'eval' / 'dada' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / root.name / 'dada'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', run_dir(root) / 'checkpoint_last.pt',
             '--t2-dir', L_T2, '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', L_T2CLIP,
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'{root.name}_dada.log')
        publish(local, out)
    r = json.loads((out / 'dada_readout.json').read_text())
    assert r['videos'] == len(DADA_IDS) and r['global_steps'] == {f's{SEED}': STEPS} and r['arm']['motion'] == OFF, r
    print(f"{root.name} DADA whole: macro {r['macro']['mean']:.4f}")

from core.tools.dota_train_extract import train_clips  # noqa: E402

# Preflight on small files only (C39)
failed_cfg, trunk_cfg = cfg_of(FAILED), cfg_of(TRUNK)
assert failed_cfg['kim.enabled'] and failed_cfg['kim.edge_mask'] and not failed_cfg.get(EF, False), 'FAILED is not L5dfiDoTA'
assert not trunk_cfg['kim.enabled'] and trunk_cfg['v2.motion'] == OFF and trunk_cfg['model.motion_dim'] == 0, 'TRUNK is not L0dfiDoTArr'
PLAN = json.loads(BUILD.read_text())['plan']
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']
assert (STEPS, EPOCHS) == (1764, 6), PLAN
for r in (FAILED, TRUNK, CAP_ROW):
    assert is_verified(run_dir(r), STEPS), f'{r.name}: J10'
TRUNK_ENV = json.loads((run_dir(TRUNK) / constants.KIM_RUN_ENV_FILENAME).read_text())
ids_all = sorted(train_clips(constants.DOTA_TRAIN_SPLITS_DIR / constants.DOTA_TRAIN_MAP_FILENAME))
DADA_IDS, CAP_IDS = test_sources(T2), sorted(load_split(CAP_DEV))
assert len(ids_all) == 3020 and len(DADA_IDS) == 370 and not set(ids_all) & set(CAP_IDS)
for name, d, ids in (('plain s3', CLIP_S3, ids_all), ('DoTA s1', DOTA_S1, CAP_IDS), ('T2 plain', T2_CLIP, DADA_IDS)):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} of {name} missing: {miss[:5]}'
print(f'preflight OK: step {STEPS} ({EPOCHS} epochs) | trunk trained on {TRUNK_ENV} | '
      f'DoTA-CAP-dev {len(CAP_IDS)} (in-domain), DADA {len(DADA_IDS)}')

## 5. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_kim_edge_fill.py',
     'core/tests/test_kim_edge_fill_report.py', 'core/tests/test_kim.py', 'core/tests/test_window_geometry_read.py',
     'core/tests/test_kim_clip_host_report.py'])

## 6. Stage the §55 corpus, plain CLIP and KNN

In [ ]:
L_DATA    = stage(DATA, LOCAL / 'stage' / 'data')
L_TRAIN_X = stage(CLIP_S3, LOCAL / 'stage' / 'plain_s3', ids_all)
L_KNN     = stage(KNN, LOCAL / 'stage' / 'knn' / KNN.name)
L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS)
L_T2CLIP  = stage(T2_CLIP, LOCAL / 'stage' / 'dada_clip', DADA_IDS)
L_LAB     = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')
assert (DATA / constants.DEFS_FILENAME).read_text() == (WINDOW / constants.DEFS_FILENAME).read_text(), 'defs differ from Nexar_window'
assert len(list(L_TRAIN_X.glob('*.npy'))) == len(ids_all), 'staged partially -- re-run this cell'
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every row read with (§58)
    shutil.copy2(f, L_DEFS / f.name)
print(f'staged: train {len(ids_all)}, DoTA-CAP-dev {len(CAP_IDS)}, DADA {len(DADA_IDS)} (same CLIP frames checked in §58 AB4)')

## 7. Train **L5dfiDoTAef**; start gate (§68 EF3/EF4)

L5dfiDoTA's exact command + `kim.edge_fill=true`. Step-1 `mil` must equal the trunk's; on another GPU or an unequal start, L0dfiDoTA's command is re-trained here as **L0dfiDoTArr2** and becomes the trunk.

In [ ]:
# L5dfiDoTA's exact command (§64 CH2) + kim.edge_fill=true (§68 EF2); the trunk = L0dfiDoTA's exact command
L0_SETS = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
           f'data.dataset={constants.DOTA_TRAIN_DATASET}', f'data.frame_stride={constants.DOTA_TRAIN_STRIDE}',
           f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}', 'kip.enabled=false', f'loss.dvs_anchor_mode={IGN}',
           'data.is_egocentric=false', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
           'dvs.theta=0.7', 'dvs.theta_ego=0.85', f'v2.motion={OFF}', 'model.motion_dim=0']
SPEC = {NEW: ([*L0_SETS, *[f'{k}=true' for k in KIM_FLAGS], f'{EF}=true'], 'ef'),
        TRUNK2.name: ([*L0_SETS, *[f'{k}=false' for k in KIM_FLAGS]], 'L0rr2')}
ROWS = {'failed': FAILED, 'trunk': TRUNK}
ROWS['fill'] = train_row(NEW, *SPEC[NEW])
env_new = json.loads((run_dir(ROWS['fill']) / constants.KIM_RUN_ENV_FILENAME).read_text())
if env_new != TRUNK_ENV or not start(TRUNK, ROWS['fill']) or is_verified(run_dir(TRUNK2), STEPS):
    print(f'EF3: env {env_new} vs trunk {TRUNK_ENV} or start differs -> re-train {TRUNK2.name}; it becomes the trunk')
    ROWS['trunk'] = train_row(TRUNK2.name, *SPEC[TRUNK2.name])
    assert start(ROWS['trunk'], ROWS['fill']), 'UNMATCHED START even against the re-run -- the report will raise; paste to Claude'
start(FAILED, ROWS['fill'])   # printed only
print('rows:', {k: v.name for k, v in ROWS.items()})

## 8. Reads: DoTA-CAP-dev (protocol B, **in-domain**) and DADA whole — plain CLIP inputs

In [ ]:
for key in ('fill', 'trunk'):
    read_dota(ROWS[key])     # skips rows already read
    read_dada(ROWS[key])
    flush_drive()

## 9. Window-geometry read of the new row (§67 tool; slot check, printed)

In [ ]:
GEO = REPORTS / 'window_geometry_ef'
local = LOCAL / 'geometry'
if not (GEO / f'{constants.KIM_WG_REPORT_STEM}.json').exists():
    shutil.rmtree(local, ignore_errors=True)
    wg_rows = {'L5': ROWS['fill'], 'L0': ROWS['trunk'], 'L5cap': CAP_ROW}
    run([sys.executable, '-m', 'core.tools.window_geometry_read',
         *[x for k, root in wg_rows.items() for x in ('--row', k, root.name, root)],
         '--s1-dir', L_DOTA_S1, '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', L_LAB, '--seed', SEED, '--device', 'cuda', '--out-dir', local], log=LOCAL / 'geometry.log')
    publish(local, GEO)
print((GEO / f'{constants.KIM_WG_REPORT_STEM}.md').read_text())

## 10. Report (§68 EF6–EF7) -> **paste `REPORTS/kim_edge_fill_report.md` to Claude**

In [ ]:
local = LOCAL / 'report'
shutil.rmtree(local, ignore_errors=True)
local.mkdir(parents=True)
run([sys.executable, '-m', 'core.tools.kim_edge_fill_report', '--seed', SEED,
     '--trunk', ROWS['trunk'].name, ROWS['trunk'], '--failed', FAILED.name, FAILED, '--row', NEW, ROWS['fill'],
     '--geometry-json', GEO / f'{constants.KIM_WG_REPORT_STEM}.json', '--out-dir', local])
publish(local, REPORTS)
stem = constants.KIM_EDGE_FILL_REPORT_STEM
print((REPORTS / f'{stem}.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f'Drive flushed -- paste REPORTS/{stem}.md to Claude')